# 1. Scrapper Tradicional

In [ ]:
# Instalar el descargador de imágenes
!pip install bing-image-downloader pillow imagehash pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 28.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import shutil
import hashlib
from pathlib import Path
from collections import defaultdict

import pandas as pd
import imagehash
from PIL import Image
from bing_image_downloader import downloader


OUTPUT_DIR = Path("dataset_got_scene_classifier")
TMP_DIR = OUTPUT_DIR / "_tmp"

LIMIT_PER_QUERY = 20

MIN_WIDTH = 384
MIN_HEIGHT = 256

# quitar posters/banner extremos
MAX_ASPECT_RATIO = 2.0
MIN_ASPECT_RATIO = 0.55

PHASH_DISTANCE_THRESHOLD = 6

DELETE_TMP_AT_END = True
VERBOSE = True


QUERIES_BY_HOUSE = {
    "Stark": [
        "Winterfell scene Game of Thrones",
        "Jon Snow Winterfell scene",
        "Arya Stark scene Game of Thrones",
        "Sansa Stark Winterfell still",
        "Ned Stark scene Game of Thrones",
        "Bran Stark Winterfell scene",
        "Stark family Winterfell",
        "The North Stark scene",
        "Stark battle scene",
        "Winterfell digital painting",
        "Stark illustration scene",
    ],
    "Lannister": [
        "Lannister scene Game of Thrones",
        "Cersei throne room scene",
        "Jaime Lannister scene",
        "Tyrion Lannister court scene",
        "Kings Landing Lannister scene",
        "Red Keep Lannister scene",
        "Lannister family scene",
        "Lannister army scene",
        "Lannister battle scene",
        "Lannister illustration scene",
        "Kings Landing digital painting",
    ],
    "Targaryen": [
        "Targaryen scene Game of Thrones",
        "Daenerys dragon scene",
        "Daenerys throne room scene",
        "Dragonstone Targaryen scene",
        "Daenerys army scene",
        "Targaryen fire scene",
        "Targaryen battle scene",
        "Daenerys illustration scene",
        "Dragonstone digital painting",
        "Targaryen cinematic still",
    ],
    "Baratheon": [
        "Baratheon scene Game of Thrones",
        "Robert Baratheon scene",
        "Stannis Baratheon scene",
        "Renly Baratheon scene",
        "Baratheon war camp scene",
        "Baratheon battle scene",
        "Baratheon cinematic still",
        "Stannis illustration scene",
        "Baratheon digital painting",
    ],
    "Greyjoy": [
        "Greyjoy scene Game of Thrones",
        "Theon Greyjoy scene",
        "Yara Greyjoy scene",
        "Iron Islands Greyjoy scene",
        "Greyjoy ship scene",
        "Greyjoy sea battle scene",
        "Ironborn Greyjoy scene",
        "Greyjoy illustration scene",
        "Iron Islands digital painting",
    ],
}

# filtro por nombre/ruta
BAD_FILENAME_KEYWORDS = [
    "shirt", "tshirt", "t-shirt", "hoodie", "sweatshirt",
    "merch", "merchandise", "funko", "toy", "figure", "collectible",
    "cosplay", "costume", "logo", "sigil", "crest", "emblem",
    "poster", "wallpaper", "mug", "sticker", "decal", "patch",
    "etsy", "amazon", "ebay", "redbubble", "tee", "print",
    "phone-case", "phone_case", "case", "ring", "necklace",
    "wearing", "outfit", "apparel", "gift"
]


def log(msg):
    if VERBOSE:
        print(msg)


def safe_name(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[-\s]+", "_", text)
    return text[:60]  # muy importante: limitar longitud


def ensure_house_dirs(house: str):
    base = OUTPUT_DIR / house
    dirs = {
        "base": base,
        "accepted": base / "accepted",
        "rejected_name": base / "rejected_name",
        "rejected_small": base / "rejected_small",
        "rejected_aspect": base / "rejected_aspect",
        "duplicates_exact": base / "duplicates_exact",
        "duplicates_visual": base / "duplicates_visual",
        "raw": base / "raw",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def compute_md5(path: Path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def image_ok_and_size(path: Path):
    try:
        with Image.open(path) as img:
            w, h = img.size
            img.verify()
        return True, w, h
    except Exception:
        return False, 0, 0


def get_phash(path: Path):
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            return imagehash.phash(img)
    except Exception:
        return None


def bad_name(path: Path):
    text = str(path).lower()
    return any(k in text for k in BAD_FILENAME_KEYWORDS)


def suspicious_aspect_ratio(w: int, h: int):
    if h == 0:
        return True
    ratio = w / h
    return ratio > MAX_ASPECT_RATIO or ratio < MIN_ASPECT_RATIO


def next_output_name(folder: Path, suffix: str):
    next_id = len(list(folder.glob("*"))) + 1
    return folder / f"{next_id:05d}{suffix.lower()}"


def clean_empty_dirs(path: Path):
    if not path.exists():
        return
    for root, dirs, files in os.walk(path, topdown=False):
        p = Path(root)
        if not dirs and not files:
            try:
                p.rmdir()
            except Exception:
                pass


def find_downloaded_images(src_dir: Path):
    allowed_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    files = []
    for p in src_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in allowed_exts:
            files.append(p)
    return files


def safe_bing_download(query: str, tmp_query_dir: Path):
    """
    Ejecuta bing_image_downloader sin tumbar el notebook aunque haga sys.exit().
    """
    try:
        downloader.download(
            query=query,
            limit=LIMIT_PER_QUERY,
            output_dir=str(tmp_query_dir),
            adult_filter_off=True,
            force_replace=False,
            timeout=30,
            filter="photo",
            verbose=False,
        )
        return True
    except SystemExit:
        # La librería hace sys.exit(1) internamente.
        log(f"  [WARN] SystemExit al descargar query: {query}")
        return False
    except Exception as e:
        log(f"  [WARN] Error descargando '{query}': {e}")
        return False


# FILTRADO
def process_downloaded_images(
    src_dir: Path,
    house: str,
    house_dirs: dict,
    exact_hashes: set,
    visual_hashes: list,
    metadata_rows: list,
    source_query: str,
    stats: dict,
):
    for file_path in find_downloaded_images(src_dir):
        stats["seen"] += 1

        # guardar raw
        try:
            raw_target = house_dirs["raw"] / file_path.name
            if not raw_target.exists():
                shutil.copy2(file_path, raw_target)
        except Exception:
            pass

        # nombre sospechoso
        if bad_name(file_path):
            stats["rejected_name"] += 1
            try:
                target = next_output_name(house_dirs["rejected_name"], file_path.suffix)
                shutil.move(str(file_path), str(target))
            except Exception:
                pass
            continue

        # validar imagen
        ok, w, h = image_ok_and_size(file_path)
        if not ok:
            stats["invalid"] += 1
            continue

        # tamaño
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            stats["rejected_small"] += 1
            try:
                target = next_output_name(house_dirs["rejected_small"], file_path.suffix)
                shutil.move(str(file_path), str(target))
            except Exception:
                pass
            continue

        # proporción
        if suspicious_aspect_ratio(w, h):
            stats["rejected_aspect"] += 1
            try:
                target = next_output_name(house_dirs["rejected_aspect"], file_path.suffix)
                shutil.move(str(file_path), str(target))
            except Exception:
                pass
            continue

        # duplicado exacto
        try:
            md5 = compute_md5(file_path)
        except Exception:
            stats["invalid"] += 1
            continue

        if md5 in exact_hashes:
            stats["duplicates_exact"] += 1
            try:
                target = next_output_name(house_dirs["duplicates_exact"], file_path.suffix)
                shutil.move(str(file_path), str(target))
            except Exception:
                pass
            continue

        # duplicado visual
        ph = get_phash(file_path)
        if ph is None:
            stats["invalid"] += 1
            continue

        is_visual_dup = False
        for existing_ph in visual_hashes:
            if abs(ph - existing_ph) <= PHASH_DISTANCE_THRESHOLD:
                is_visual_dup = True
                break

        if is_visual_dup:
            stats["duplicates_visual"] += 1
            try:
                target = next_output_name(house_dirs["duplicates_visual"], file_path.suffix)
                shutil.move(str(file_path), str(target))
            except Exception:
                pass
            continue

        # aceptar
        exact_hashes.add(md5)
        visual_hashes.append(ph)

        try:
            target = next_output_name(house_dirs["accepted"], file_path.suffix)
            shutil.move(str(file_path), str(target))
            stats["accepted"] += 1

            metadata_rows.append({
                "house": house,
                "filepath": str(target),
                "query": source_query,
                "width": w,
                "height": h,
                "aspect_ratio": round(w / h, 4),
            })
        except Exception:
            stats["invalid"] += 1


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    global_stats = defaultdict(lambda: {
        "queries": 0,
        "seen": 0,
        "accepted": 0,
        "rejected_name": 0,
        "rejected_small": 0,
        "rejected_aspect": 0,
        "duplicates_exact": 0,
        "duplicates_visual": 0,
        "invalid": 0,
        "download_fail": 0,
    })

    metadata_rows = []

    for house, queries in QUERIES_BY_HOUSE.items():
        log("\n" + "=" * 80)
        log(f"CASA: {house}")
        log("=" * 80)

        house_dirs = ensure_house_dirs(house)

        exact_hashes = set()
        visual_hashes = []

        # cargar aceptadas previas si reejecutas
        for existing in house_dirs["accepted"].glob("*"):
            if existing.is_file():
                try:
                    exact_hashes.add(compute_md5(existing))
                except Exception:
                    pass
                try:
                    ph = get_phash(existing)
                    if ph is not None:
                        visual_hashes.append(ph)
                except Exception:
                    pass

        for i, query in enumerate(queries, start=1):
            global_stats[house]["queries"] += 1
            log(f"\n[{house}] Query {i}/{len(queries)}")
            log(query)

            tmp_query_dir = TMP_DIR / f"{safe_name(house)}_{i:02d}_{safe_name(query)}"
            tmp_query_dir.mkdir(parents=True, exist_ok=True)

            ok = safe_bing_download(query, tmp_query_dir)
            if not ok:
                global_stats[house]["download_fail"] += 1
                continue

            process_downloaded_images(
                src_dir=tmp_query_dir,
                house=house,
                house_dirs=house_dirs,
                exact_hashes=exact_hashes,
                visual_hashes=visual_hashes,
                metadata_rows=metadata_rows,
                source_query=query,
                stats=global_stats[house],
            )

            clean_empty_dirs(tmp_query_dir)

        log(f"\nResumen {house}:")
        for k, v in global_stats[house].items():
            log(f"  {k}: {v}")

    pd.DataFrame(metadata_rows).to_csv(
        OUTPUT_DIR / "metadata.csv",
        index=False,
        encoding="utf-8"
    )

    if DELETE_TMP_AT_END and TMP_DIR.exists():
        try:
            shutil.rmtree(TMP_DIR)
        except Exception:
            pass

    print("\n" + "=" * 80)
    print("RESUMEN FINAL")
    print("=" * 80)
    for house, stats in global_stats.items():
        print(
            f"{house:12} | "
            f"queries={stats['queries']:2d} | "
            f"accepted={stats['accepted']:4d} | "
            f"rej_name={stats['rejected_name']:4d} | "
            f"rej_small={stats['rejected_small']:4d} | "
            f"rej_aspect={stats['rejected_aspect']:4d} | "
            f"dup_exact={stats['duplicates_exact']:4d} | "
            f"dup_visual={stats['duplicates_visual']:4d} | "
            f"download_fail={stats['download_fail']:3d} | "
            f"invalid={stats['invalid']:4d}"
        )

    print("\nDataset guardado en:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()


CASA: Stark

[Stark] Query 1/11
Winterfell scene Game of Thrones
[%] Downloading Images to /content/dataset_got_scene_classifier/_tmp/stark_01_winterfell_scene_game_of_thrones/Winterfell scene Game of Thrones
[!] Issue getting: https://i.redd.it/jna2r68kbws21.jpg
[!] Error:: HTTP Error 403: Blocked
[!] Issue getting: https://thirdeyetraveller.com/wp-content/uploads/Castle-Ward-Game-of-Thrones-real-Winterfell-Castle-10.jpg
[!] Error:: HTTP Error 520: <none>
[!] Issue getting: https://www.indiependent.co.uk/wp-content/uploads/2019/04/GOT.jpg
[!] Error:: HTTP Error 403: Forbidden


[%] Done. Downloaded 20 images.

[Stark] Query 2/11
Jon Snow Winterfell scene
[%] Downloading Images to /content/dataset_got_scene_classifier/_tmp/stark_02_jon_snow_winterfell_scene/Jon Snow Winterfell scene
[Error]Invalid image, not saving https://64.media.tumblr.com/6645c0306fa15aac8e168142c9572e96/bb12dd9a584abc14-a8/s1280x1920/461b8c1d17f93157f6fdedbcd14c75b237d43fd8.png

[!] Issue getting: https://64.medi

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



[Lannister] Query 6/11
Red Keep Lannister scene
[%] Downloading Images to /content/dataset_got_scene_classifier/_tmp/lannister_06_red_keep_lannister_scene/Red Keep Lannister scene
[!] Issue getting: https://www.mindtaker.org/wp-content/uploads/2023/09/IMG_6191-600x600.jpg
[!] Error:: HTTP Error 403: Forbidden
[Error]Invalid image, not saving https://64.media.tumblr.com/e8cddac422f8e1bd14593ae527cec6e0/tumblr_ptvme5cQnz1t6gzreo1_500.jpg

[!] Issue getting: https://64.media.tumblr.com/e8cddac422f8e1bd14593ae527cec6e0/tumblr_ptvme5cQnz1t6gzreo1_500.jpg
[!] Error:: Invalid image, not saving https://64.media.tumblr.com/e8cddac422f8e1bd14593ae527cec6e0/tumblr_ptvme5cQnz1t6gzreo1_500.jpg

[!] Issue getting: https://preview.redd.it/cersei-lannister-sunset-in-at-the-red-keep-by-me-v0-gc85zqc1fz5a1.jpg?width=640&amp;crop=smart&amp;auto=webp&amp;s=f88a403b1411f047f441bb74414beb47419153b4
[!] Error:: HTTP Error 403: Blocked
[!] Issue getting: https://preview.redd.it/my-take-on-the-lannister-red-c

Ahora zipeamos el dataset para descargarlo

In [ ]:
!zip -r dataset_got_scene_classifier.zip /content/dataset_got_scene_classifier

  adding: content/dataset_got_scene_classifier/ (stored 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/ (stored 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/ (stored 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00006.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00052.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00163.jpg (deflated 1%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00135.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00161.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00083.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00117.jpg (deflated 1%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00080.jpg (deflated 0%)
  adding: content/dataset_got_scene_classifier/Targaryen/accepted/00047

# 2. Scrapper por temas de casa

In [1]:
!pip install -q bing-image-downloader

In [2]:
import os
import shutil
from bing_image_downloader import downloader
from google.colab import files

# 2. Nuestro Superdiccionario de Escenas
diccionario_casas_got = {
    'Baratheon': [
        "King Robert Baratheon arrival at Winterfell retinue Game of Thrones",
        "Robert Baratheon exploring Winterfell crypts with Ned Stark",
        "Joffrey Baratheon sparring with Robb Stark wooden swords",
        "Joffrey Baratheon walking with Sansa Stark near the Trident",
        "Renly Baratheon green armor Tourney of the Hand",
        "King Robert Baratheon deathbed after boar hunt Game of Thrones"
    ],
    'Greyjoy': [
        "Theon Greyjoy kicking severed head in the snow Game of Thrones",
        "Theon Greyjoy holding bow saving Bran from wildlings",
        "Theon Greyjoy standing behind Robb Stark calling banners",
        "Theon Greyjoy riding beside Robb Stark army marching south",
        "Theon Greyjoy fighting in the Whispering Wood battle"
    ],
    'Lannister': [
        "Tyrion Lannister reading a book in Winterfell library",
        "Jaime Lannister pushing Bran Stark from the broken tower window",
        "Tyrion Lannister slapping Joffrey Baratheon in Winterfell",
        "Cersei Lannister confronting Ned Stark in the Red Keep Godswood",
        "Tyrion Lannister descending into the sky cells of the Eyrie",
        "Tywin Lannister war council at the Crossroads Inn",
        "Jaime Lannister captured by Robb Stark Whispering Wood",
        "Tyrion Lannister leading mountain clans Battle of the Green Fork"
    ],
    'Stark': [
        "Ned Stark executing Gared Night's Watch deserter Ice sword",
        "Stark children finding the direwolf pups in the summer snow",
        "Arya Stark practicing water dancing with Syrio Forel",
        "Jon Snow taking his oath at the Weirwood tree beyond the Wall",
        "Catelyn Stark arresting Tyrion Lannister at the Crossroads Inn",
        "Sansa Stark pleading for her father's life Iron Throne",
        "Ned Stark fighting Jaime Lannister guards in King's Landing streets",
        "Robb Stark declared King in the North by bannermen"
    ],
    'Targaryen': [
        "Daenerys Targaryen meeting Khal Drogo in Pentos manse",
        "Daenerys Targaryen receiving three dragon eggs wedding gift",
        "Viserys Targaryen demanding his crown molten gold",
        "Daenerys Targaryen eating raw horse heart in Vaes Dothrak",
        "Daenerys Targaryen blood magic tent Mirri Maz Duur",
        "Maester Aemon revealing his Targaryen identity to Jon Snow",
        "Daenerys Targaryen stepping into roaring funeral pyre",
        "Daenerys Targaryen unburnt nursing three baby dragons ashes"
    ]
}

# 3. Configuración
directorio_base = "Dataset_GOT_Definitivo_V3"

# Función auxiliar para sacar las imágenes de las subcarpetas y ponerlas en la carpeta de la casa
def mover_imagenes_al_directorio_padre(query_folder_name, casa_dir, prefijo_nombre):
    subcarpeta = os.path.join(casa_dir, query_folder_name)
    if os.path.exists(subcarpeta):
        for index, filename in enumerate(os.listdir(subcarpeta)):
            ruta_antigua = os.path.join(subcarpeta, filename)
            # Limpiamos el prefijo para que sea un nombre de archivo válido
            nombre_limpio = "".join(x for x in prefijo_nombre if x.isalnum() or x in " ").replace(" ", "_")[:40]
            extension = os.path.splitext(filename)[1]
            nuevo_nombre = f"{nombre_limpio}_{index+1}{extension}"
            ruta_nueva = os.path.join(casa_dir, nuevo_nombre)

            # Evitar sobreescribir si ya existe un archivo con ese nombre
            contador = 1
            while os.path.exists(ruta_nueva):
                nuevo_nombre = f"{nombre_limpio}_{index+1}_{contador}{extension}"
                ruta_nueva = os.path.join(casa_dir, nuevo_nombre)
                contador += 1

            shutil.move(ruta_antigua, ruta_nueva)

        # Eliminamos la subcarpeta vacía
        os.rmdir(subcarpeta)

# 4. Proceso de Descarga con Bing
for casa, escenas in diccionario_casas_got.items():
    print(f"\n{'='*50}")
    print(f"🛡️ DESCARGANDO CASA: {casa.upper()} 🛡️")
    print(f"{'='*50}")

    # Carpeta principal de la casa
    ruta_casa = os.path.join(directorio_base, casa)
    os.makedirs(ruta_casa, exist_ok=True)

    for escena in escenas:
        print(f"Buscando escena: {escena[:40]}...")

        # --- BÚSQUEDA 1: ARTE E ILUSTRACIÓN (5 imágenes) ---
        query_art = escena + " illustration art concept"
        try:
            downloader.download(query_art, limit=5, output_dir=ruta_casa, adult_filter_off=True, force_replace=False, timeout=60, verbose=False)
            mover_imagenes_al_directorio_padre(query_art, ruta_casa, escena + "_ART")
            print(f"  🎨 +5 Arte descargado.")
        except Exception as e:
            print(f"  ❌ Error con el arte de '{escena}': {e}")

        # --- BÚSQUEDA 2: ESCENAS REALES HBO (3 imágenes) ---
        query_real = escena + " HBO show live action scene"
        try:
            downloader.download(query_real, limit=3, output_dir=ruta_casa, adult_filter_off=True, force_replace=False, timeout=60, verbose=False)
            mover_imagenes_al_directorio_padre(query_real, ruta_casa, escena + "_REAL")
            print(f"  🎬 +3 Escenas reales descargadas.")
        except Exception as e:
            print(f"  ❌ Error con escena real de '{escena}': {e}")

# 5. Comprimir y descargar
print("\n📦 Comprimiendo el dataset en un archivo ZIP...")
shutil.make_archive(directorio_base, 'zip', directorio_base)
print("🚀 ¡Todo listo! Lanzando la descarga a tu ordenador...")

try:
    files.download(f"{directorio_base}.zip")
except:
    print(f"👉 Si no se descarga automáticamente, haz clic derecho en '{directorio_base}.zip' en el panel de la izquierda y dale a 'Download'.")


🛡️ DESCARGANDO CASA: BARATHEON 🛡️
Buscando escena: King Robert Baratheon arrival at Winterf...
[%] Downloading Images to /content/Dataset_GOT_Definitivo_V3/Baratheon/King Robert Baratheon arrival at Winterfell retinue Game of Thrones illustration art concept
[!] Issue getting: https://i0.wp.com/www.paolopuggioni.com/admin/wp-content/uploads/2012/07/King-Robert-Baratheon.jpg?ssl=1
[!] Error:: HTTP Error 404: Not Found
[!] Issue getting: https://wallpapercrafter.com/desktop/103467-Game-of-Thrones-Robert-Baratheon-knight-horse-war-Warhammer-sword.jpg
[!] Error:: HTTP Error 403: Forbidden


[%] Done. Downloaded 5 images.
  🎨 +5 Arte descargado.
[%] Downloading Images to /content/Dataset_GOT_Definitivo_V3/Baratheon/King Robert Baratheon arrival at Winterfell retinue Game of Thrones HBO show live action scene
[!] Issue getting: https://i0.wp.com/www.paolopuggioni.com/admin/wp-content/uploads/2012/07/King-Robert-Baratheon.jpg?resize=600%2C772&amp;ssl=1
[!] Error:: HTTP Error 404: Not Found



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>